# 12 — Chevron Experiments

Consolidates legacy experiments 27, 28, and 29.

1. **Time Rabi Chevron** — 2D sweep (detuning × duration)
2. **Power Rabi Chevron** — 2D sweep (detuning × gain)
3. **Ramsey Chevron** — 2D sweep (frequency × wait time)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="12_chevron_experiments",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

coherence_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="06_coherence_experiments",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
if coherence_checkpoint is not None:
    print(f"Prior stage 06 status: {coherence_checkpoint['status']}")

## 2. Chevron Defaults

In [ ]:
# --- Time Rabi Chevron (Exp 27) ---
TIME_RABI_FREQ_START = float(attr.qb_fq) - 50 * u.MHz
TIME_RABI_FREQ_END = float(attr.qb_fq) + 50 * u.MHz
TIME_RABI_DF = 1 * u.MHz
TIME_RABI_MAX_DURATION_NS = 200
TIME_RABI_DT_NS = 4
TIME_RABI_N_AVG = 1000

# --- Power Rabi Chevron (Exp 28) ---
POWER_RABI_FREQ_START = float(attr.qb_fq) - 50 * u.MHz
POWER_RABI_FREQ_END = float(attr.qb_fq) + 50 * u.MHz
POWER_RABI_DF = 1 * u.MHz
POWER_RABI_MAX_GAIN = 1.5
POWER_RABI_DG = 0.05
POWER_RABI_N_AVG = 1000

# --- Ramsey Chevron (Exp 29) ---
RAMSEY_FREQ_START = float(attr.qb_fq) - 20 * u.MHz
RAMSEY_FREQ_END = float(attr.qb_fq) + 20 * u.MHz
RAMSEY_DF = 500 * u.kHz
RAMSEY_MAX_WAIT_NS = 2000
RAMSEY_DT_NS = 20
RAMSEY_N_AVG = 1000

print("Chevron experiment settings:")
print(f"  Time Rabi: Δf=±50 MHz, max_t={TIME_RABI_MAX_DURATION_NS} ns, n_avg={TIME_RABI_N_AVG}")
print(f"  Power Rabi: Δf=±50 MHz, max_gain={POWER_RABI_MAX_GAIN}, n_avg={POWER_RABI_N_AVG}")
print(f"  Ramsey: Δf=±20 MHz, max_wait={RAMSEY_MAX_WAIT_NS} ns, n_avg={RAMSEY_N_AVG}")

## 3. Time Rabi Chevron — Exp 27

2D sweep: qubit drive frequency × pulse duration.

In [ ]:
time_rabi_chev_result = session.time_rabi_chevron(
    freq_start=TIME_RABI_FREQ_START,
    freq_end=TIME_RABI_FREQ_END,
    df=TIME_RABI_DF,
    max_duration_ns=TIME_RABI_MAX_DURATION_NS,
    dt_ns=TIME_RABI_DT_NS,
    n_avg=TIME_RABI_N_AVG,
)

print("Time Rabi chevron complete.")

## 4. Power Rabi Chevron — Exp 28

2D sweep: qubit drive frequency × pulse amplitude.

In [ ]:
power_rabi_chev_result = session.power_rabi_chevron(
    freq_start=POWER_RABI_FREQ_START,
    freq_end=POWER_RABI_FREQ_END,
    df=POWER_RABI_DF,
    max_gain=POWER_RABI_MAX_GAIN,
    dg=POWER_RABI_DG,
    n_avg=POWER_RABI_N_AVG,
)

print("Power Rabi chevron complete.")

## 5. Ramsey Chevron — Exp 29

2D sweep: qubit drive frequency × Ramsey wait time.

In [ ]:
ramsey_chev_result = session.ramsey_chevron(
    freq_start=RAMSEY_FREQ_START,
    freq_end=RAMSEY_FREQ_END,
    df=RAMSEY_DF,
    max_wait_ns=RAMSEY_MAX_WAIT_NS,
    dt_ns=RAMSEY_DT_NS,
    n_avg=RAMSEY_N_AVG,
)

print("Ramsey chevron complete.")

## 6. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="12_chevron_experiments",
    status="characterized",
    summary="Ran Time Rabi, Power Rabi, and Ramsey chevron 2D sweeps for qubit characterization.",
    consumed_inputs={
        "time_rabi_n_avg": TIME_RABI_N_AVG,
        "power_rabi_n_avg": POWER_RABI_N_AVG,
        "ramsey_n_avg": RAMSEY_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="13_dispersive_shift_measurement",
    notes=[
        "These are characterization-only experiments — no calibration patches are applied.",
        "Ramsey chevron (Exp 29) was flagged as a possible copy-paste duplicate in legacy — verify results.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")